# Questão 02: Pós-Treino por SFT Completo (docentesDC)

Ajuste fino supervisionado completo do **Qwen2.5-0.5B** com pares de instrução gerados programaticamente do dataset docentesDC. Contém o *setup* compartilhado (Seção 0) seguido da Questão 02. Saídas preservadas da execução validada no Kaggle.


## Seção 0: Setup global

A célula abaixo precisa ser a primeira célula de código do notebook: `CUDA_VISIBLE_DEVICES` limita a sessão a uma única GPU antes de qualquer `import torch`. Com as duas T4 visíveis, o Trainer ativaria DataParallel e o gather dos logits (vocabulário de 151k tokens) estoura a memória da GPU 0. O modelo de 0.5B cabe com folga em uma única T4.

In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import time
import random

MODO_TESTE = False
RODAR_Q1 = True
RODAR_BONUS_15B = True

SEED = 42
MAX_LEN = 256
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
MODEL_NAME_BONUS = 'Qwen/Qwen2.5-1.5B'
T0 = time.time()

LIMITE_BONUS_H = 9.5
LIMITE_TETO_H = 11.0

if MODO_TESTE:
    Q1_MAX_DOCS = 300
    Q1_N_AVALIACAO = 50
    Q1_EPOCAS = 1
    Q2_ALVO_PARES = 120
    Q2_EPOCAS = 1
    Q3_EPOCAS = 1
    BONUS_MAX_STEPS = 10
    N_BENCHMARK_Q1 = 5
else:
    Q1_MAX_DOCS = None
    Q1_N_AVALIACAO = 1000
    Q1_EPOCAS = 2
    Q2_ALVO_PARES = 2000
    Q2_EPOCAS = 3
    Q3_EPOCAS = 3
    BONUS_MAX_STEPS = -1
    N_BENCHMARK_Q1 = 25


def tempo_decorrido_h():
    return (time.time() - T0) / 3600


random.seed(SEED)
print(f'MODO_TESTE={MODO_TESTE} | RODAR_Q1={RODAR_Q1} | RODAR_BONUS_15B={RODAR_BONUS_15B}')
print(f'Modelo principal: {MODEL_NAME} | MAX_LEN={MAX_LEN} | SEED={SEED}')

### Instalação segura de dependências

A stack do Kaggle (torch, transformers 5.x, datasets, accelerate) fica intocada: reinstalar qualquer um desses pacotes foi o que quebrou os imports nas primeiras tentativas da Questão 01. Só `peft` e `bitsandbytes` são instalados, com `--no-deps`, e apenas se estiverem ausentes. `peft` ausente é erro fatal imediato (a Questão 03 depende dele); `bitsandbytes` ausente apenas desativa o caminho QLoRA (o LoRA puro cumpre a questão).

In [ ]:
import importlib.util
import subprocess
import sys


def garantir_pacote(nome):
    if importlib.util.find_spec(nome) is None:
        print(f'Instalando {nome} com --no-deps...')
        resultado = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', nome],
            capture_output=True, text=True)
        if resultado.returncode != 0:
            print(resultado.stderr[-800:])
        importlib.invalidate_caches()


garantir_pacote('peft')
garantir_pacote('bitsandbytes')

PEFT_OK = False
BNB_OK = False
try:
    from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
    import peft
    PEFT_OK = True
except Exception as erro:
    print(f'peft indisponível: {erro}')
try:
    import bitsandbytes
    BNB_OK = True
except Exception as erro:
    print(f'bitsandbytes indisponível: {erro}')

if not PEFT_OK:
    raise RuntimeError('peft é obrigatório para a Questão 3 e não pôde ser carregado. Verifique se a Internet do notebook está ligada.')

import torch
import transformers

print(f'torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}')
print(f'PEFT_OK={PEFT_OK} | BNB_OK={BNB_OK}')

In [ ]:
import gc
import json
import math
import re
import shutil
import hashlib
import traceback
import unicodedata
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling,
    default_data_collator,
)

try:
    from transformers import BitsAndBytesConfig
except Exception as erro:
    print(f'BitsAndBytesConfig indisponível: {erro}')
    BNB_OK = False

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    assert torch.cuda.device_count() == 1, 'Esperada exatamente 1 GPU visível'
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'Disco livre: {shutil.disk_usage(".").free / 1e9:.1f} GB')

### Funções compartilhadas

Definidas uma única vez e usadas pelas três questões: salvamento de artefatos, estilo e helpers de gráfico, montagem de prompts no formato Alpaca e limpeza de memória entre seções.

In [ ]:
DIR_FIGURAS = 'figuras'
os.makedirs(DIR_FIGURAS, exist_ok=True)

COR_ANTES = '#b5651d'
COR_DEPOIS = '#12805a'
C_FULL = '#b5651d'
C_LORA = '#2a78d6'
C_QLORA = '#12805a'
C_BONUS = '#4a3aa7'
C_BASE = '#898781'
C_EDA = '#2a78d6'
COR_GRADE = '#e1e0d9'

plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'figure.facecolor': 'white',
    'axes.grid': False,
    'axes.axisbelow': True,
})

RESULTADOS = {}


def salvar_json(objeto, caminho):
    conteudo = json.dumps(objeto, ensure_ascii=False, indent=2, default=str)
    with open(caminho, 'w', encoding='utf-8') as arquivo:
        arquivo.write(conteudo)
    print(f'Salvo: {caminho}')
    if len(conteudo) < 100000:
        print(f'----- backup em log: inicio de {caminho} -----')
        print(conteudo)
        print(f'----- backup em log: fim de {caminho} -----')


def salvar_fig(fig, nome):
    caminho = os.path.join(DIR_FIGURAS, nome)
    fig.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Figura salva: {caminho}')


def rotular_barras(ax, barras, sufixo='', casas=2):
    for barra in barras:
        altura = barra.get_height()
        ax.annotate(f'{altura:.{casas}f}{sufixo}',
                    xy=(barra.get_x() + barra.get_width() / 2, altura),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')


def grafico_metricas_multi(series, titulo, nome_arquivo):
    paineis = [('Entropia Cruzada', 'entropia_cruzada', '', 3),
               ('Perplexidade', 'perplexidade', '', 2),
               ('Acurácia de Tokens (%)', 'acuracia_tokens', '%', 1)]
    largura = 12 + max(0, len(series) - 2) * 1.4
    fig, eixos = plt.subplots(1, 3, figsize=(largura, 4.2))
    for ax, (nome, chave, sufixo, casas) in zip(eixos, paineis):
        rotulos = [s[0] for s in series]
        valores = [s[1][chave] for s in series]
        cores = [s[2] for s in series]
        barras = ax.bar(rotulos, valores, color=cores, width=0.62)
        rotular_barras(ax, barras, sufixo=sufixo, casas=casas)
        ax.set_title(nome)
        ax.set_ylim(0, max(valores) * 1.28 + 1e-9)
        ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
        if len(series) > 2:
            ax.tick_params(axis='x', labelrotation=20)
    fig.suptitle(titulo, fontweight='bold')
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_barh(itens, titulo, nome_arquivo, cor=None, log_x=False, fmt='{:,.0f}'):
    nomes = [str(item[0]) for item in itens]
    valores = [item[1] for item in itens]
    fig, ax = plt.subplots(figsize=(9.5, max(3, 0.42 * len(itens) + 1.2)))
    barras = ax.barh(nomes[::-1], valores[::-1], color=cor or C_EDA)
    if log_x:
        ax.set_xscale('log')
    for barra in barras:
        largura = barra.get_width()
        ax.annotate(fmt.format(largura),
                    xy=(largura, barra.get_y() + barra.get_height() / 2),
                    xytext=(4, 0), textcoords='offset points',
                    va='center', fontsize=9)
    ax.set_title(titulo)
    ax.grid(axis='x', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_hist_comprimentos(comprimentos, cortes, titulo, nome_arquivo, mediana=None):
    dados = [c for c in comprimentos if c > 0]
    bins = np.logspace(np.log10(max(min(dados), 1)), np.log10(max(dados)), 50)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(dados, bins=bins, color=C_EDA, edgecolor='white', linewidth=0.3)
    ax.set_xscale('log')
    for corte in cortes:
        ax.axvline(corte, color='#8a1d1d', linestyle='--', linewidth=1.2)
    if mediana:
        ax.axvline(mediana, color='#4a3aa7', linestyle=':', linewidth=1.4)
        ax.annotate(f'mediana {mediana:,.0f}', xy=(mediana, ax.get_ylim()[1] * 0.9),
                    xytext=(6, 0), textcoords='offset points', fontsize=9, color='#4a3aa7')
    ax.set_xlabel('Comprimento (caracteres, escala log)')
    ax.set_ylabel('Documentos')
    ax.set_title(titulo)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_curva_loss(log_history, titulo, nome_arquivo):
    pontos_treino = [(h['step'], h['loss']) for h in log_history if 'loss' in h and 'eval_loss' not in h]
    pontos_eval = [(h['step'], h['eval_loss']) for h in log_history if 'eval_loss' in h]
    fig, ax = plt.subplots(figsize=(9, 4.2))
    if pontos_treino:
        xs, ys = zip(*pontos_treino)
        ax.plot(xs, ys, color=C_EDA, linewidth=1.6, label='Loss de treino')
    if pontos_eval:
        xs, ys = zip(*pontos_eval)
        ax.plot(xs, ys, color='#184f95', marker='o', linestyle='--', linewidth=1.2, label='Loss de avaliação')
    ax.set_xlabel('Passo')
    ax.set_ylabel('Loss')
    ax.set_title(titulo)
    ax.legend(frameon=False)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_curvas_loss(series, titulo, nome_arquivo):
    fig, ax = plt.subplots(figsize=(9, 4.2))
    for rotulo, log_history, cor in series:
        pontos = [(h['step'], h['loss']) for h in log_history if 'loss' in h and 'eval_loss' not in h]
        if pontos:
            xs, ys = zip(*pontos)
            ax.plot(xs, ys, color=cor, linewidth=1.6, label=rotulo)
    ax.set_xlabel('Passo')
    ax.set_ylabel('Loss de treino')
    ax.set_title(titulo)
    ax.legend(frameon=False)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_painel_comparativo(entidades, titulo, nome_arquivo):
    paineis = [('Perplexidade (depois)', 'ppl', '', 2),
               ('Acurácia de Tokens (depois, %)', 'acc', '%', 1),
               ('Tempo de treino (min)', 'tempo_min', '', 1),
               ('Pico de VRAM (GB)', 'vram', '', 2)]
    fig, eixos = plt.subplots(2, 2, figsize=(11, 8))
    for ax, (nome, chave, sufixo, casas) in zip(eixos.flat, paineis):
        rotulos = [e['rotulo'] for e in entidades]
        valores = [e[chave] for e in entidades]
        cores = [e['cor'] for e in entidades]
        barras = ax.bar(rotulos, valores, color=cores, width=0.6)
        rotular_barras(ax, barras, sufixo=sufixo, casas=casas)
        ax.set_title(nome)
        ax.set_ylim(0, max(valores) * 1.28 + 1e-9)
        ax.tick_params(axis='x', labelrotation=15)
        ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.suptitle(titulo, fontweight='bold')
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def montar_prompt(instrucao, entrada=''):
    if entrada and entrada.strip():
        return f'### Instrução:\n{instrucao}\n\n### Entrada:\n{entrada}\n\n### Resposta:\n'
    return f'### Instrução:\n{instrucao}\n\n### Resposta:\n'


def normalizar_texto(texto):
    texto = unicodedata.normalize('NFKD', texto.lower())
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]', '', texto)


def taxa_acerto_por_categoria(resultados):
    por_categoria = {}
    for r in resultados:
        acertou = normalizar_texto(r['resposta_esperada']) in normalizar_texto(r['resposta_gerada'])
        ok, total = por_categoria.get(r['cat'], (0, 0))
        por_categoria[r['cat']] = (ok + (1 if acertou else 0), total + 1)
    return {c: round(ok / total * 100, 1) for c, (ok, total) in por_categoria.items()}


def atualizar_zip_parcial():
    os.makedirs('parciais', exist_ok=True)
    for nome in os.listdir('.'):
        if nome.endswith('.json'):
            shutil.copy(nome, os.path.join('parciais', nome))
    if os.path.isdir(DIR_FIGURAS):
        shutil.copytree(DIR_FIGURAS, os.path.join('parciais', DIR_FIGURAS), dirs_exist_ok=True)
    shutil.make_archive('resultados_parciais', 'zip', 'parciais')
    print('Zip parcial atualizado: resultados_parciais.zip')


print('Helpers de artefatos e gráficos definidos.')

In [ ]:
def carregar_modelo_fp32(nome):
    try:
        modelo = AutoModelForCausalLM.from_pretrained(nome, dtype=torch.float32)
    except TypeError:
        modelo = AutoModelForCausalLM.from_pretrained(nome, torch_dtype=torch.float32)
    return modelo.to(DEVICE)


def montar_training_args(**kwargs):
    for tentativa in range(6):
        try:
            return TrainingArguments(**kwargs)
        except (TypeError, ValueError) as erro:
            mensagem = str(erro)
            removido = None
            for chave in list(kwargs):
                if chave in mensagem:
                    removido = chave
                    kwargs.pop(chave)
                    break
            if removido is None:
                raise
            print(f'Argumento removido por incompatibilidade de versão: {removido}')
    raise RuntimeError('Não foi possível montar TrainingArguments')


def calcular_metricas(modelo, tok, textos, prompts=None, batch_size=8):
    modelo.eval()
    total_loss = 0.0
    total_validos = 0
    total_certos = 0
    for inicio in range(0, len(textos), batch_size):
        lote = textos[inicio:inicio + batch_size]
        enc = tok(lote, return_tensors='pt', max_length=MAX_LEN, truncation=True, padding=True)
        input_ids = enc['input_ids'].to(DEVICE)
        attention_mask = enc['attention_mask'].to(DEVICE)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        if prompts is not None:
            for j, prompt in enumerate(prompts[inicio:inicio + batch_size]):
                n_prompt = len(tok(prompt)['input_ids'])
                labels[j, :min(n_prompt, labels.shape[1])] = -100
        with torch.no_grad():
            saida = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        mascara_validos = labels[:, 1:] != -100
        n_validos = mascara_validos.sum().item()
        if n_validos == 0:
            continue
        total_loss += saida.loss.item() * n_validos
        total_validos += n_validos
        predicoes = saida.logits.argmax(dim=-1)
        certos = ((predicoes[:, :-1] == input_ids[:, 1:]) & mascara_validos).sum().item()
        total_certos += certos
    entropia = total_loss / max(total_validos, 1)
    return {'entropia_cruzada': round(entropia, 4),
            'perplexidade': round(math.exp(min(entropia, 20)), 2),
            'acuracia_tokens': round(total_certos / max(total_validos, 1) * 100, 2)}


class SFTDatasetMasked(Dataset):
    def __init__(self, pares, tok):
        self.exemplos = []
        for prompt, completo in pares:
            ids_prompt = tok(prompt)['input_ids']
            ids_completo = tok(completo)['input_ids']
            ids_resposta = ids_completo[len(ids_prompt):]
            espaco_resposta = MAX_LEN - len(ids_prompt)
            if espaco_resposta < 2:
                continue
            if len(ids_resposta) > espaco_resposta:
                ids_resposta = ids_resposta[:espaco_resposta - 1] + [tok.eos_token_id]
            input_ids = ids_prompt + ids_resposta
            n_pad = MAX_LEN - len(input_ids)
            attention_mask = [1] * len(input_ids) + [0] * n_pad
            labels = [-100] * len(ids_prompt) + list(ids_resposta) + [-100] * n_pad
            input_ids = input_ids + [tok.pad_token_id] * n_pad
            self.exemplos.append({
                'input_ids': torch.tensor(input_ids),
                'attention_mask': torch.tensor(attention_mask),
                'labels': torch.tensor(labels),
            })

    def __len__(self):
        return len(self.exemplos)

    def __getitem__(self, indice):
        return self.exemplos[indice]


def gerar_resposta(modelo, tok, instrucao, entrada='', max_new_tokens=90):
    modelo.eval()
    prompt = montar_prompt(instrucao, entrada)
    entradas = tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(saida[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def gerar_texto_livre(modelo, tok, prompt, max_new_tokens=80):
    modelo.eval()
    entradas = tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=True,
                                temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                                pad_token_id=tok.eos_token_id)
    return tok.decode(saida[0], skip_special_tokens=True)


def rodar_benchmark(modelo, tok, itens, max_new_tokens=100):
    modelo.eval()
    resultados = []
    for item in itens:
        entradas = tok(item['pergunta'], return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tok.pad_token_id or tok.eos_token_id)
        gerada = tok.decode(saida[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        resultados.append({'id': item['id'], 'cat': item['cat'], 'pergunta': item['pergunta'],
                           'resposta_esperada': item['resposta'], 'resposta_gerada': gerada})
    return resultados


def responder_qualitativa(modelo, tok, itens):
    respostas = []
    for item in itens:
        resposta = gerar_resposta(modelo, tok, item['instruction'], item.get('input', ''))
        respostas.append({'id': item['id'], 'resposta': resposta})
    return respostas


def liberar_gpu(*nomes):
    espaco = globals()
    for nome in nomes:
        espaco.pop(nome, None)
    gc.collect()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f'VRAM alocada: {torch.cuda.memory_allocated() / 1e9:.2f} GB | reservada: {torch.cuda.memory_reserved() / 1e9:.2f} GB')


def status_recursos(rotulo):
    print(f'[{rotulo}] tempo decorrido: {tempo_decorrido_h():.2f}h | disco livre: {shutil.disk_usage(".").free / 1e9:.1f} GB')
    if torch.cuda.is_available():
        print(f'[{rotulo}] VRAM alocada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')


def remover_checkpoints(diretorio):
    if os.path.isdir(diretorio):
        shutil.rmtree(diretorio, ignore_errors=True)
        print(f'Checkpoints removidos: {diretorio}')


print('Helpers de modelo, métricas e memória definidos.')

### Pré-voo

Valida em poucos minutos tudo que poderia falhar depois de horas de treino: acesso aos dois datasets do Hugging Face com os campos esperados, tokenizer, GPU única e disco. Se algo estiver errado, o notebook morre aqui, no minuto 3, e não na hora 6.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer: {MODEL_NAME} | vocabulário {tokenizer.vocab_size:,}')

verificacao_docentes = load_dataset('vickminari/docentesDC', split='train')
assert 'text' in verificacao_docentes.column_names, 'Campo text ausente no docentesDC'
assert 'nome_professor' in verificacao_docentes.column_names, 'Campo nome_professor ausente no docentesDC'
print(f'docentesDC ok: {len(verificacao_docentes):,} linhas | colunas {verificacao_docentes.column_names}')
del verificacao_docentes

verificacao_dompi = load_dataset('gutoportelaa/DOMPI-2025', name='raw', split='entre_rios')
assert 'texto' in verificacao_dompi.column_names, 'Campo texto ausente no DOMPI-2025'
print(f'DOMPI-2025 ok (split entre_rios): {len(verificacao_dompi):,} documentos')
del verificacao_dompi
gc.collect()

livre_gb = shutil.disk_usage('.').free / 1e9
if os.path.isdir('/kaggle'):
    assert livre_gb > 15, f'Disco livre insuficiente: {livre_gb:.1f} GB'

status_recursos('pré-voo concluído')

## Seção 2: Questão 02, pós-treino por SFT completo (docentesDC)

O `docentesDC` tem 13.762 linhas com apenas dois campos (`text` e `nome_professor`, 19 professores) e conteúdo bastante ruidoso: código C cru, papers acadêmicos em inglês fatiados em chunks e slides com formatação quebrada. Como não há um LLM gerador disponível, todos os `output` dos pares são derivados programaticamente do próprio dado e são factualmente verdadeiros por construção (nada de resposta inventada como gabarito).

Tipos de pares gerados (alvo de ~2.000, mínimo do enunciado é 1.000):

| Tipo | Tarefa | Racional |
|---|---|---|
| A | atribuição de autoria (trecho no input) | resposta função determinística do contexto |
| B | continuação do trecho (só português) | ensina o estilo do material |
| C | tema do trecho (por mapa de palavras-chave, 2+ ocorrências) | rotulagem verificável |
| D | fatos fechados sobre o dataset (sem input) | fatos que o modelo base não conhece, demo qualitativa forte |
| E | código C: autoria e linguagem | aproveita o volume de código sem inventar explicações |

Um held-out de 10% dos chunks por professor é separado ANTES da geração: os pares de avaliação saem só de chunks held-out, para a perplexidade "depois" não medir memorização do trecho visto em treino.

In [ ]:
print('Carregando docentesDC do Hugging Face...')
ds_docentes = load_dataset('vickminari/docentesDC', split='train')
docentes_textos = ds_docentes['text']
docentes_profs = ds_docentes['nome_professor']
del ds_docentes
contagem_professores = Counter(docentes_profs)
print(f'{len(docentes_textos):,} linhas | {len(contagem_professores)} professores')

grafico_barh(contagem_professores.most_common(), 'docentesDC: linhas por professor', 'q2_eda_professores.png')
comprimentos_docentes = [len(t) if t else 0 for t in docentes_textos]
mediana_docentes = float(np.median([c for c in comprimentos_docentes if c > 0]))
grafico_hist_comprimentos(comprimentos_docentes, [200, 20000],
                          'docentesDC: distribuição de comprimento dos textos',
                          'q2_eda_comprimentos.png', mediana=mediana_docentes)

In [ ]:
def limpar_chunk(texto):
    texto = re.sub(r'[\u00d8\u2022\u25aa\u25e6\ue000-\uf8ff]', '\n', texto)
    texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', texto)
    texto = re.sub(r'^\s*\d{1,3}\s+', '', texto)
    texto = re.sub(r' {2,}', ' ', texto)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()


MARCAS_CODIGO = ['#include', 'int main', 'printf(', 'scanf(', 'void ', 'struct ', 'malloc', 'return 0;', '};']
STOP_EN = {'the', 'of', 'and', 'to', 'in', 'is', 'that', 'for', 'with', 'are', 'this', 'on', 'as', 'by', 'we', 'be'}
STOP_PT = {'de', 'que', 'do', 'da', 'em', 'um', 'uma', 'para', 'com', 'não', 'os', 'dos', 'por', 'mais', 'como', 'é', 'são', 'foi', 'pelo'}


def classificar_chunk(texto):
    baixo = texto.lower()
    hits_codigo = sum(1 for marca in MARCAS_CODIGO if marca in baixo)
    densidade = sum(1 for c in texto if c in '{};()=<>') / max(len(texto), 1)
    if hits_codigo >= 2 or densidade > 0.04:
        return 'codigo'
    palavras = baixo.split()
    hits_en = sum(1 for p in palavras if p in STOP_EN)
    hits_pt = sum(1 for p in palavras if p in STOP_PT)
    if hits_en >= 8 and hits_en / (hits_en + hits_pt + 1) > 0.6:
        return 'ingles'
    return 'pt'


def assinaturas_chunk(texto):
    norma = re.sub(r'\d+', '', texto.lower())
    norma = re.sub(r'\s+', ' ', norma).strip()
    tokens = norma.split()
    inteiro = hashlib.md5(norma.encode()).hexdigest()
    inicio = hashlib.md5(' '.join(tokens[:60]).encode()).hexdigest()
    fim = hashlib.md5(' '.join(tokens[-60:]).encode()).hexdigest()
    return inteiro, inicio, fim


def recortar(texto, max_chars):
    if len(texto) <= max_chars:
        return texto.strip()
    trecho = texto[:max_chars]
    corte = trecho.rfind(' ')
    if corte > max_chars * 0.6:
        trecho = trecho[:corte]
    return trecho.strip()


MAPA_TOPICOS = {
    'Estruturas de Dados': ['pilha', 'fila', 'árvore binária', 'grafo', 'lista encadeada', 'ordenação', 'busca binária'],
    'Redes de Computadores': ['tcp', 'socket', 'roteamento', 'ieee 802', 'protocolo', 'ethernet', 'endereço ip'],
    'Banco de Dados': ['sql', 'normalização', 'chave primária', 'modelo relacional', 'consulta', 'tupla'],
    'Sistemas Operacionais': ['escalonamento', 'deadlock', 'memória virtual', 'sistema operacional', 'semáforo', 'thread'],
    'Arquitetura de Computadores': ['registrador', 'pipeline', 'memória cache', 'barramento', 'processador', 'ula'],
    'Inteligência Artificial': ['rede neural', 'aprendizado de máquina', 'agente', 'classificador', 'inteligência artificial'],
    'Engenharia de Software': ['requisitos', 'uml', 'caso de uso', 'diagrama de classes', 'engenharia de software'],
}


def inferir_topico(texto):
    baixo = texto.lower()
    melhor = None
    melhor_hits = 0
    for topico, chaves in MAPA_TOPICOS.items():
        hits = sum(1 for chave in chaves if chave in baixo)
        if hits > melhor_hits:
            melhor = topico
            melhor_hits = hits
    if melhor_hits >= 2:
        return melhor
    return None


print('Funções do pipeline de pares definidas.')

In [ ]:
funil = {'brutos': len(docentes_textos)}
chunks_docentes = []
for texto, professor in zip(docentes_textos, docentes_profs):
    if not texto:
        continue
    limpo = limpar_chunk(texto)
    if not (200 <= len(limpo) <= 20000):
        continue
    chunks_docentes.append({'texto': limpo, 'professor': professor.title(), 'classe': classificar_chunk(limpo)})
funil['apos_limpeza_e_tamanho'] = len(chunks_docentes)

assinaturas_vistas = set()
chunks_unicos = []
for chunk in chunks_docentes:
    sigs = assinaturas_chunk(chunk['texto'])
    if any(s in assinaturas_vistas for s in sigs):
        continue
    assinaturas_vistas.update(sigs)
    chunks_unicos.append(chunk)
funil['apos_dedup'] = len(chunks_unicos)

contagem_classes = Counter(c['classe'] for c in chunks_unicos)
funil['portugues'] = contagem_classes.get('pt', 0)
funil['ingles'] = contagem_classes.get('ingles', 0)
funil['codigo'] = contagem_classes.get('codigo', 0)

random.seed(SEED)
por_professor = defaultdict(list)
for chunk in chunks_unicos:
    por_professor[chunk['professor']].append(chunk)

CAP_POR_PROFESSOR = 200
chunks_treino = []
chunks_heldout = []
for professor in sorted(por_professor):
    lista = por_professor[professor]
    random.shuffle(lista)
    lista = lista[:CAP_POR_PROFESSOR]
    n_heldout = max(1, int(len(lista) * 0.1))
    chunks_heldout.extend(lista[:n_heldout])
    chunks_treino.extend(lista[n_heldout:])
funil['apos_amostragem'] = len(chunks_treino) + len(chunks_heldout)

print(json.dumps(funil, ensure_ascii=False, indent=2))
print(f'Chunks para treino: {len(chunks_treino):,} | held-out para avaliação: {len(chunks_heldout):,}')
del docentes_textos, docentes_profs, chunks_docentes, assinaturas_vistas
gc.collect()

In [ ]:
TEMPLATES_A = [
    'Qual professor do Departamento de Computação da UFPI é o autor do material a seguir?',
    'Identifique o docente do DC/UFPI responsável por este trecho de material.',
    'A qual professor do DC/UFPI pertence o material apresentado?',
    'Informe o nome do professor autor deste trecho de material didático.',
]
TEMPLATES_B = [
    'Continue o trecho a seguir do material do professor {prof}.',
    'Complete a sequência deste material de aula do professor {prof}.',
    'Dê continuidade ao texto a seguir, extraído do material do professor {prof}.',
    'Prossiga o conteúdo deste trecho do material do professor {prof}.',
]
TEMPLATES_C = [
    'Sobre qual assunto trata este trecho do material do professor {prof}?',
    'Qual é o tema abordado neste trecho do material do professor {prof}?',
    'Identifique o assunto principal deste trecho de material do professor {prof}.',
    'De qual tema trata o material a seguir, do professor {prof}?',
]
TEMPLATES_E_AUTOR = [
    'A qual professor pertence este material de aula com código?',
    'Qual docente do DC/UFPI é o autor deste material com código?',
    'Identifique o professor responsável por este material de aula que contém código.',
    'Informe o professor autor do material com código a seguir.',
]
TEMPLATES_E_LINGUAGEM = [
    'Em qual linguagem de programação está escrito o código a seguir?',
    'Identifique a linguagem de programação do código apresentado.',
    'Qual linguagem de programação foi usada no código a seguir?',
    'Diga em qual linguagem o código apresentado foi escrito.',
]


def par_tipo_a(chunk):
    return {'instruction': random.choice(TEMPLATES_A), 'input': recortar(chunk['texto'], 450),
            'output': f"Este material pertence ao professor {chunk['professor']}.",
            'tipo': 'A', 'professor': chunk['professor']}


def par_tipo_b(chunk):
    prefixo = recortar(chunk['texto'], 250)
    resto = chunk['texto'][len(prefixo):].strip()
    continuacao = recortar(resto, 400)
    if len(continuacao) < 100:
        return None
    return {'instruction': random.choice(TEMPLATES_B).format(prof=chunk['professor']),
            'input': prefixo, 'output': continuacao, 'tipo': 'B', 'professor': chunk['professor']}


def par_tipo_c(chunk):
    topico = inferir_topico(chunk['texto'])
    if topico is None:
        return None
    return {'instruction': random.choice(TEMPLATES_C).format(prof=chunk['professor']),
            'input': recortar(chunk['texto'], 450),
            'output': f"O trecho trata de {topico}, tema abordado no material do professor {chunk['professor']}.",
            'tipo': 'C', 'professor': chunk['professor']}


def par_tipo_e(chunk, com_linguagem):
    if com_linguagem and '#include' in chunk['texto']:
        return {'instruction': random.choice(TEMPLATES_E_LINGUAGEM), 'input': recortar(chunk['texto'], 350),
                'output': 'O código está escrito em linguagem C.', 'tipo': 'E', 'professor': chunk['professor']}
    return {'instruction': random.choice(TEMPLATES_E_AUTOR), 'input': recortar(chunk['texto'], 350),
            'output': f"Este material com código pertence ao professor {chunk['professor']}.",
            'tipo': 'E', 'professor': chunk['professor']}


def gerar_pares(chunks, alvo_por_tipo, rotulo_split):
    pares = []
    pool_pt = [c for c in chunks if c['classe'] == 'pt']
    pool_codigo = [c for c in chunks if c['classe'] == 'codigo']
    pool_todos = list(chunks)
    for pool in (pool_pt, pool_codigo, pool_todos):
        random.shuffle(pool)
    gerados = 0
    for chunk in pool_todos:
        if gerados >= alvo_por_tipo['A']:
            break
        pares.append(par_tipo_a(chunk))
        gerados += 1
    gerados = 0
    for chunk in pool_pt:
        if gerados >= alvo_por_tipo['B']:
            break
        par = par_tipo_b(chunk)
        if par:
            pares.append(par)
            gerados += 1
    gerados = 0
    for chunk in pool_pt:
        if gerados >= alvo_por_tipo['C']:
            break
        par = par_tipo_c(chunk)
        if par:
            pares.append(par)
            gerados += 1
    gerados = 0
    for indice, chunk in enumerate(pool_codigo * 2):
        if gerados >= alvo_por_tipo['E']:
            break
        pares.append(par_tipo_e(chunk, com_linguagem=(indice % 2 == 1)))
        gerados += 1
    for par in pares:
        par['split'] = rotulo_split
    return pares


def gerar_pares_tipo_d(quantidade):
    professores = sorted({c['professor'] for c in chunks_unicos})
    professor_top = contagem_professores.most_common(1)[0][0].title()
    topicos_professores = defaultdict(set)
    for chunk in chunks_treino:
        topico = inferir_topico(chunk['texto'])
        if topico:
            topicos_professores[topico].add(chunk['professor'])
    candidatos = [
        ('Quem é o professor com mais material no dataset docentesDC?',
         f'O professor com mais material no dataset docentesDC é {professor_top}.'),
        ('Qual docente possui o maior volume de material no dataset docentesDC?',
         f'O docente com maior volume de material no docentesDC é {professor_top}.'),
        ('Diga o nome do professor com mais registros no dataset docentesDC.',
         f'O professor com mais registros no docentesDC é {professor_top}.'),
        ('Quantos professores compõem o dataset docentesDC?',
         f'O dataset docentesDC reúne materiais de {len(professores)} professores do Departamento de Computação da UFPI.'),
        ('Quantos docentes estão presentes no dataset docentesDC e de onde eles são?',
         f'O docentesDC contém materiais de {len(professores)} professores do Departamento de Computação (DC) da UFPI.'),
        ('De qual universidade e departamento são os professores do dataset docentesDC?',
         'Os professores do docentesDC são do Departamento de Computação (DC) da Universidade Federal do Piauí (UFPI).'),
        ('O dataset docentesDC reúne materiais de professores de qual instituição?',
         'O docentesDC reúne materiais de professores da Universidade Federal do Piauí (UFPI), do Departamento de Computação.'),
    ]
    for _ in range(12):
        trio = random.sample(professores, 3)
        candidatos.append(('Cite três professores do Departamento de Computação da UFPI presentes no dataset docentesDC.',
                           f'Três professores presentes no dataset docentesDC são {trio[0]}, {trio[1]} e {trio[2]}.'))
    for topico in sorted(topicos_professores):
        profs_topico = ', '.join(sorted(topicos_professores[topico])[:4])
        candidatos.append((f'Quais professores do DC/UFPI possuem materiais sobre {topico}?',
                           f'Possuem materiais sobre {topico} os professores {profs_topico}.'))
    for professor in professores:
        candidatos.append((f'O professor {professor} está presente no dataset docentesDC?',
                           f'Sim, o professor {professor} é um dos docentes com material no dataset docentesDC.'))
    random.shuffle(candidatos)
    n_candidatos_eval = max(2, len(candidatos) // 10)
    candidatos_eval = candidatos[:n_candidatos_eval]
    candidatos_treino = candidatos[n_candidatos_eval:]
    pares_eval = [{'instruction': instrucao, 'input': '', 'output': saida, 'tipo': 'D',
                   'professor': '', 'split': 'eval'}
                  for instrucao, saida in candidatos_eval]
    pares_treino = []
    indice = 0
    alvo_treino_d = max(0, quantidade - len(pares_eval))
    while len(pares_treino) < alvo_treino_d and candidatos_treino:
        instrucao, saida = candidatos_treino[indice % len(candidatos_treino)]
        pares_treino.append({'instruction': instrucao, 'input': '', 'output': saida, 'tipo': 'D',
                             'professor': '', 'split': 'treino'})
        indice += 1
    return pares_treino, pares_eval


FRACOES_TIPOS = {'A': 0.35, 'B': 0.25, 'C': 0.20, 'E': 0.10}
alvo_treino = {tipo: max(2, int(Q2_ALVO_PARES * 0.9 * fracao)) for tipo, fracao in FRACOES_TIPOS.items()}
alvo_eval = {tipo: max(2, int(Q2_ALVO_PARES * 0.1 * fracao)) for tipo, fracao in FRACOES_TIPOS.items()}

random.seed(SEED)
pares_treino = gerar_pares(chunks_treino, alvo_treino, 'treino')
pares_eval = gerar_pares(chunks_heldout, alvo_eval, 'eval')
pares_d_treino, pares_d_eval = gerar_pares_tipo_d(max(4, int(Q2_ALVO_PARES * 0.10)))
pares_treino.extend(pares_d_treino)
pares_eval.extend(pares_d_eval)

dados_sft = pares_treino + pares_eval
funil['pares_gerados'] = len(dados_sft)
salvar_json(dados_sft, 'dados_sft_docentesDC.json')

contagem_tipos = Counter(par['tipo'] for par in dados_sft)
print(f'Pares gerados: {len(dados_sft):,} | por tipo: {dict(sorted(contagem_tipos.items()))}')
print(f"Treino: {sum(1 for p in dados_sft if p['split'] == 'treino'):,} | Avaliação: {sum(1 for p in dados_sft if p['split'] == 'eval'):,}")
for tipo in 'ABCDE':
    exemplo = next((p for p in dados_sft if p['tipo'] == tipo), None)
    if exemplo:
        print(f"\n[Tipo {tipo}] {exemplo['instruction']}")
        if exemplo['input']:
            print(f"  input : {exemplo['input'][:120]}")
        print(f"  output: {exemplo['output'][:160]}")

etapas_funil = [('Registros brutos', funil['brutos']),
                ('Após limpeza e tamanho', funil['apos_limpeza_e_tamanho']),
                ('Após deduplicação', funil['apos_dedup']),
                ('Após amostragem por professor', funil['apos_amostragem']),
                ('Pares gerados', funil['pares_gerados'])]
grafico_barh(etapas_funil, 'Questão 2: funil do pipeline de geração de pares', 'q2_pipeline_funil.png', cor='#4a3aa7')

tipos_ordenados = sorted(contagem_tipos.items())
fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar([t for t, _ in tipos_ordenados], [v for _, v in tipos_ordenados], color=C_EDA, width=0.6)
rotular_barras(ax, barras, casas=0)
ax.set_title('Questão 2: pares gerados por tipo')
ax.set_xlabel('Tipo de par')
ax.set_ylabel('Quantidade')
ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
fig.tight_layout()
salvar_fig(fig, 'q2_tipos_pares.png')

### Conjunto de avaliação qualitativa (10 perguntas)

Perguntas com professores reais do dataset, respondidas por todos os modelos (base, Full SFT, LoRA, QLoRA e bônus) com geração greedy, para uma comparação lado a lado reprodutível. Os trechos de input vêm exclusivamente de chunks held-out.

In [ ]:
def primeiro_chunk(pool, condicao):
    for chunk in pool:
        if condicao(chunk):
            return chunk
    return None


professores_dataset = sorted({c['professor'] for c in chunks_unicos})
professor_top_nome = contagem_professores.most_common(1)[0][0].title()
contagem_topicos = Counter()
for chunk in chunks_treino:
    topico = inferir_topico(chunk['texto'])
    if topico:
        contagem_topicos[topico] += 1

avaliacao_qualitativa = [
    {'tipo': 'D', 'instruction': 'Quem é o professor com mais material no dataset docentesDC?',
     'input': '', 'referencia': professor_top_nome},
    {'tipo': 'D', 'instruction': 'Cite três professores do Departamento de Computação da UFPI presentes no dataset docentesDC.',
     'input': '', 'referencia': 'Qualquer trio de: ' + ', '.join(professores_dataset)},
    {'tipo': 'D', 'instruction': 'Quantos professores compõem o dataset docentesDC e de qual departamento eles são?',
     'input': '', 'referencia': f'{len(professores_dataset)} professores do Departamento de Computação da UFPI'},
]
if contagem_topicos:
    topico_mais_comum = contagem_topicos.most_common(1)[0][0]
    profs_do_topico = sorted({c['professor'] for c in chunks_treino if inferir_topico(c['texto']) == topico_mais_comum})[:4]
    avaliacao_qualitativa.append({'tipo': 'D',
                                  'instruction': f'Quais professores do DC/UFPI possuem materiais sobre {topico_mais_comum}?',
                                  'input': '', 'referencia': ', '.join(profs_do_topico)})

chunks_usados = set()


def adicionar_item_com_chunk(tipo, condicao, instrucao_fn, referencia_fn, tam_input=450):
    chunk = primeiro_chunk(chunks_heldout, lambda c: condicao(c) and id(c) not in chunks_usados)
    if chunk is None:
        return False
    chunks_usados.add(id(chunk))
    avaliacao_qualitativa.append({'tipo': tipo, 'instruction': instrucao_fn(chunk),
                                  'input': recortar(chunk['texto'], tam_input),
                                  'referencia': referencia_fn(chunk)})
    return True


adicionar_item_com_chunk('A', lambda c: 'Erico' in c['professor'] and c['classe'] != 'codigo',
                         lambda c: TEMPLATES_A[0], lambda c: c['professor'])
adicionar_item_com_chunk('A', lambda c: 'Ivan' in c['professor'] and c['classe'] != 'codigo',
                         lambda c: TEMPLATES_A[1], lambda c: c['professor'])
while sum(1 for item in avaliacao_qualitativa if item['tipo'] == 'A') < 2:
    if not adicionar_item_com_chunk('A', lambda c: c['classe'] != 'codigo',
                                    lambda c: TEMPLATES_A[2], lambda c: c['professor']):
        break

adicionar_item_com_chunk('E', lambda c: c['classe'] == 'codigo',
                         lambda c: TEMPLATES_E_AUTOR[0], lambda c: c['professor'], tam_input=350)
adicionar_item_com_chunk('C', lambda c: c['classe'] == 'pt' and inferir_topico(c['texto']) is not None and 'Vinicius' in c['professor'],
                         lambda c: TEMPLATES_C[0].format(prof=c['professor']), lambda c: inferir_topico(c['texto']))
adicionar_item_com_chunk('C', lambda c: c['classe'] == 'pt' and inferir_topico(c['texto']) is not None,
                         lambda c: TEMPLATES_C[1].format(prof=c['professor']), lambda c: inferir_topico(c['texto']))

chunk_b = primeiro_chunk(chunks_heldout, lambda c: 'Raimundo' in c['professor'] and c['classe'] == 'pt' and len(c['texto']) >= 700 and id(c) not in chunks_usados)
if chunk_b is None:
    chunk_b = primeiro_chunk(chunks_heldout, lambda c: c['classe'] == 'pt' and len(c['texto']) >= 700 and id(c) not in chunks_usados)
if chunk_b is not None:
    chunks_usados.add(id(chunk_b))
    prefixo_b = recortar(chunk_b['texto'], 250)
    resto_b = chunk_b['texto'][len(prefixo_b):].strip()
    avaliacao_qualitativa.append({'tipo': 'B',
                                  'instruction': TEMPLATES_B[0].format(prof=chunk_b['professor']),
                                  'input': prefixo_b, 'referencia': recortar(resto_b, 400)})

for numero, item in enumerate(avaliacao_qualitativa, start=1):
    item['id'] = numero
salvar_json(avaliacao_qualitativa, 'avaliacao_qualitativa_docentes.json')
print(f'{len(avaliacao_qualitativa)} perguntas de avaliação qualitativa salvas.')
for item in avaliacao_qualitativa:
    print(f"[{item['id']}][{item['tipo']}] {item['instruction']}")

### Formatação Alpaca, máscara de loss e datasets

Cada par vira um prompt Alpaca. Os labels são pré-computados com -100 em todo o prompt (até o fim de `### Resposta:\n`) e no padding, então a loss (de treino e de avaliação) mede apenas os tokens de resposta. Por isso o collator é o `default_data_collator`: o `DataCollatorForLanguageModeling` recriaria os labels a partir dos input_ids e descartaria a máscara silenciosamente. Pares cujo prompt sozinho não deixa espaço para a resposta dentro de MAX_LEN são descartados.

In [ ]:
def formatar_par(exemplo):
    prompt = montar_prompt(exemplo['instruction'], exemplo.get('input', ''))
    return prompt, prompt + exemplo['output'] + tokenizer.eos_token


def prompt_cabe(prompt):
    return len(tokenizer(prompt)['input_ids']) <= MAX_LEN - 32


sft_pares_treino = []
sft_pares_eval = []
descartados_prompt_longo = 0
for exemplo in dados_sft:
    prompt, completo = formatar_par(exemplo)
    if not prompt_cabe(prompt):
        descartados_prompt_longo += 1
        continue
    if exemplo['split'] == 'eval':
        sft_pares_eval.append((prompt, completo))
    else:
        sft_pares_treino.append((prompt, completo))

random.seed(SEED)
random.shuffle(sft_pares_treino)
sft_textos_eval = [par[1] for par in sft_pares_eval]
sft_prompts_eval = [par[0] for par in sft_pares_eval]
print(f'Descartados por prompt longo: {descartados_prompt_longo}')
print(f'Treino: {len(sft_pares_treino):,} | Avaliação: {len(sft_pares_eval):,}')
print('Exemplo formatado:')
print(sft_pares_treino[0][1][:500])

In [ ]:
sft_train_ds = SFTDatasetMasked(sft_pares_treino, tokenizer)
sft_eval_ds = SFTDatasetMasked(sft_pares_eval, tokenizer)
print(f'Datasets tokenizados com máscara de prompt: {len(sft_train_ds):,} treino | {len(sft_eval_ds):,} avaliação')

In [ ]:
model_base = carregar_modelo_fp32(MODEL_NAME)
print('Métricas do modelo base, calculadas somente nos tokens de resposta...')
q2_metricas_antes = calcular_metricas(model_base, tokenizer, sft_textos_eval, prompts=sft_prompts_eval)
print(f'ANTES do SFT: {q2_metricas_antes}')

print('\nRespostas do modelo base às perguntas de avaliação qualitativa:')
q2_respostas_antes = responder_qualitativa(model_base, tokenizer, avaliacao_qualitativa)
for item, resposta in zip(avaliacao_qualitativa, q2_respostas_antes):
    print(f"[{item['id']}] {item['instruction']}")
    print(f"    antes: {resposta['resposta'][:140]}")
liberar_gpu('model_base')

### Treinamento do Full SFT

Micro-batch 2 com acumulação 8 (batch efetivo 16), o mesmo usado nos treinos LoRA/QLoRA da Questão 03, para a comparação de pico de VRAM entre técnicas ser justa. O tempo, o pico de VRAM e o número de parâmetros treinados são registrados para os gráficos comparativos.

In [ ]:
model_ft = carregar_modelo_fp32(MODEL_NAME)
model_ft.config.use_cache = False
q2_params_totais = sum(p.numel() for p in model_ft.parameters())
OUTPUT_DIR_Q2 = 'qwen25_sft_docentes'
args_q2 = montar_training_args(
    output_dir=OUTPUT_DIR_Q2,
    num_train_epochs=Q2_EPOCAS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=0.05,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=(DEVICE.type == 'cuda'),
    dataloader_num_workers=0,
    logging_steps=20,
    disable_tqdm=True,
    report_to='none',
)
if DEVICE.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
trainer_q2 = Trainer(model=model_ft, args=args_q2, train_dataset=sft_train_ds,
                     eval_dataset=sft_eval_ds, data_collator=default_data_collator)
print(f'Iniciando Full SFT ({Q2_EPOCAS} épocas, batch efetivo 16)...')
resultado_treino_q2 = trainer_q2.train()
q2_tempo_treino = resultado_treino_q2.metrics['train_runtime']
q2_vram_pico = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == 'cuda' else 0.0
q2_log_history = [dict(h) for h in trainer_q2.state.log_history]
q2_loss_final = resultado_treino_q2.training_loss
print(f'Treino concluído: loss final {q2_loss_final:.4f} | tempo {q2_tempo_treino:.0f}s | pico de VRAM {q2_vram_pico:.2f} GB')

In [ ]:
grafico_curva_loss(q2_log_history, 'Questão 2: curva de loss do Full SFT', 'q2_curva_loss.png')
model_ft.config.use_cache = True
print('Métricas após o SFT, somente nos tokens de resposta...')
q2_metricas_depois = calcular_metricas(model_ft, tokenizer, sft_textos_eval, prompts=sft_prompts_eval)
print(f'DEPOIS do SFT: {q2_metricas_depois}')
grafico_metricas_multi(
    [('Antes', q2_metricas_antes, COR_ANTES), ('Depois', q2_metricas_depois, COR_DEPOIS)],
    'Questão 2: métricas antes e depois do Full SFT (tokens de resposta)', 'q2_metricas.png')

q2_respostas_depois = responder_qualitativa(model_ft, tokenizer, avaliacao_qualitativa)
for item, antes, depois in zip(avaliacao_qualitativa, q2_respostas_antes, q2_respostas_depois):
    print(f"[{item['id']}][{item['tipo']}] {item['instruction']}")
    print(f"    referência: {str(item['referencia'])[:110]}")
    print(f"    antes     : {antes['resposta'][:110]}")
    print(f"    depois    : {depois['resposta'][:110]}")
    print('-' * 90)

In [ ]:
qualitativa_q2 = []
for item, antes, depois in zip(avaliacao_qualitativa, q2_respostas_antes, q2_respostas_depois):
    qualitativa_q2.append({'id': item['id'], 'tipo': item['tipo'], 'instruction': item['instruction'],
                           'referencia': item['referencia'], 'resposta_antes': antes['resposta'],
                           'resposta_depois': depois['resposta']})

resultados_sft_q2 = {
    'modelo': MODEL_NAME,
    'dataset': 'vickminari/docentesDC',
    'n_treino': len(sft_pares_treino),
    'n_avaliacao': len(sft_pares_eval),
    'hiperparametros': {'epocas': Q2_EPOCAS, 'micro_batch': 2, 'grad_accum': 8, 'batch_efetivo': 16,
                        'learning_rate': 5e-05, 'max_length': MAX_LEN, 'mascara_loss': True},
    'pipeline': funil,
    'tipos_pares': dict(sorted(contagem_tipos.items())),
    'metricas_antes': q2_metricas_antes,
    'metricas_depois': q2_metricas_depois,
    'delta': {
        'entropia_cruzada': round(q2_metricas_depois['entropia_cruzada'] - q2_metricas_antes['entropia_cruzada'], 4),
        'perplexidade': round(q2_metricas_depois['perplexidade'] - q2_metricas_antes['perplexidade'], 2),
        'acuracia_tokens_pp': round(q2_metricas_depois['acuracia_tokens'] - q2_metricas_antes['acuracia_tokens'], 2),
    },
    'treino': {'tempo_s': round(q2_tempo_treino, 1), 'vram_pico_gb': round(q2_vram_pico, 2),
               'loss_final': round(q2_loss_final, 4), 'params_treinaveis': q2_params_totais,
               'params_totais': q2_params_totais},
    'qualitativa': qualitativa_q2,
    'log_history': q2_log_history,
}
salvar_json(resultados_sft_q2, 'resultados_sft_q2.json')
RESULTADOS['q2'] = resultados_sft_q2

model_ft.half()
model_ft.save_pretrained('q2_modelo_final')
tokenizer.save_pretrained('q2_modelo_final')
print('Modelo da Questão 2 salvo em fp16 em q2_modelo_final/')

remover_checkpoints(OUTPUT_DIR_Q2)
liberar_gpu('model_ft', 'trainer_q2')
atualizar_zip_parcial()
status_recursos('fim da Questão 2')